# 01 — Reproduction check, then fits (3 seeds × 28 nested fits)

`exp4_touch` recipe (`exp2_aug` rounds, provider targets, 4 touch columns, aux off, `TrainConfig()`), seeds 20261111 → 12 → 13. Each fit saves its weights and final training NLL and predicts both views of its held-out matches. Cached by fingerprint with per-job locks. Progress in `results/fit_log.txt`.

In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp5_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_5')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp5_common as C
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp5_protocol as P, exp1_net as N
print('device', N.DEVICE, '| config', N.TrainConfig().as_dict())

PREREG verified dc28ebfcacfa2676 | frozen 2026-11-10T09:06:10+00:00
device cuda | config {'window': 300, 'batch': 32, 'epochs': 14, 'coverage': 2.0, 'lr': 0.002, 'weight_decay': 0.0001, 'width': 64, 'hidden': 64, 'boundary_weight': 0.5, 'boundary_pos_weight': 3.0, 'mirror': True, 'seed': 20261108}


## Reproduction check (1 fit; must pass before any proxy fit)
exp_5's code path with `exp4_touch`'s seed 20261108, outer fold holding out J03WMX, against exp_4's cached prediction for that fold (read in place). Bit-exact expected; max |Δp| > 1e-3 stops the run.

In [2]:
log_path = C.RESULTS / 'fit_log.txt'
def log(message, **_):
    print(message, flush=True)
    with log_path.open('a') as handle:
        handle.write(message + '\n')
path = C.RESULTS / 'repro_check.json'
if path.exists():
    repro = C1.read_json(path)
else:
    t0 = time.time()
    repro = P.reproduction_check()
    repro['seconds'] = round(time.time() - t0, 1)
    C1.write_json(path, repro)
    log(f"REPRO CHECK {repro['arm']} outer_{repro['held_out']} bit_exact={repro['bit_exact']} passed={repro['passed']} {repro['seconds']}s")
print(json.dumps(repro, indent=1))
assert repro['passed'], 'reproduction check failed: stop and diagnose before any proxy fit'

{
 "arm": "exp4_touch",
 "seed": 20261108,
 "held_out": "J03WMX",
 "fits": 1,
 "exp4_cache": "exp_4/cache/predictions/exp4_touch_2d42a3706658/outer_J03WMX.npz",
 "original": {
  "max_abs_probability_diff": 0.0,
  "max_abs_boundary_diff": 0.0,
  "bit_exact": true
 },
 "target_like": {
  "max_abs_probability_diff": 0.0,
  "max_abs_boundary_diff": 0.0,
  "bit_exact": true
 },
 "bit_exact": true,
 "passed": true,
 "seconds": 52.1
}


In [3]:
t0 = time.time()
reserve = C.PLANNED_FITS['full_data'] + C.PLANNED_FITS['rebuild']
for arm in P.FIT_ORDER:
    remaining = len(P.missing_jobs(arm))
    assert P.proxy_fit_count() + 1 + remaining + reserve <= C.FIT_CAP, 'fit cap reached'
    start = time.time()
    P.ensure_predictions(arm, log=log)
    log(f'ARM DONE {arm} {time.time() - start:.0f}s, proxy fits cached {P.proxy_fit_count()}')
log(f'ALL FITS DONE {time.time() - t0:.0f}s, exp_5 proxy fits cached {P.proxy_fit_count()} (+1 reproduction check)')

ARM DONE exp5_touch 0s, proxy fits cached 29


FIT exp5_touch_s1 pair_J03WMX_J03WOY 161.2s train_nll 0.1195


FIT exp5_touch_s1 pair_J03WN1_J03WOH 149.0s train_nll 0.1241


FIT exp5_touch_s1 pair_J03WN1_J03WQQ 147.3s train_nll 0.1219


FIT exp5_touch_s1 pair_J03WOH_J03WPY 148.8s train_nll 0.1334


FIT exp5_touch_s1 pair_J03WOY_J03WPY 145.5s train_nll 0.1280


... (15 outputs left out)

FIT exp5_touch_s2 outer_J03WPY 118.3s train_nll 0.1250


FIT exp5_touch_s2 outer_J03WR9 105.6s train_nll 0.1225


ARM DONE exp5_touch_s2 1490s, proxy fits cached 84


ALL FITS DONE 2903s, exp_5 proxy fits cached 84 (+1 reproduction check)


In [4]:
train = pd.DataFrame([r for arm in P.ARMS for r in P.train_nll_rows(arm)])
print(train.groupby('arm').train_nll.agg(['mean', 'min', 'max', 'count']).round(4).to_string())

                 mean     min     max  count
arm                                         
exp5_touch     0.1221  0.1146  0.1287     28
exp5_touch_s1  0.1235  0.1157  0.1334     28
exp5_touch_s2  0.1255  0.1184  0.1333     28
